# 🎓 Trabajo Final Big Data: Éxito Académico y Deserción en Educación Superior
**Institución:** Universidad Nacional de Juliaca (UNAJ)  
**Curso:** Big Data  
**Estudiante:** joell  
**Docente:** Leibnihtz Abrahan Ayamamani Choque  

---
## Ficha del Proyecto
* **Metodología:** CRISP-DM para el flujo analítico + DAMA para la calidad de datos.
* **Dataset:** *UCI Students Dropout and Academic Success Dataset* (4,424 registros estudiantiles).
* **Entorno sugerido:** Python 3.13.14 (Kernel: `Python (EdaUnaj2026I)`)
---

## 1. CRISP-DM — Fase 1: Comprensión del Negocio

### Contexto y Motivación
La deserción universitaria representa una de las problemáticas más severas de la gestión de la educación superior, afectando la sostenibilidad financiera de las instituciones y limitando el desarrollo profesional de los estudiantes vulnerables. Predecir a tiempo qué alumnos están en alto riesgo permite coordinar intervenciones preventivas (tutorías, becas y consejería psicológicas).

### Pregunta Analítica
> *¿En qué medida la combinación de las condiciones demográficas, la vulnerabilidad socioeconómica del entorno familiar y el rendimiento académico durante el primer año (créditos matriculados, créditos aprobados y notas de los dos primeros semestres) permiten clasificar y predecir de forma temprana el riesgo de deserción en los estudiantes universitarios?*

### Justificación de Negocio
Construir un clasificador preventivo permite una detección pasiva a partir de datos ya registrados en los sistemas de matrícula al inicio del año académico (bajo costo operativo), evitando encuestas físicas masivas o la detección tardía cuando el alumno ya abandonó las aulas de forma irreversible.

## 2. Configuración del Entorno y Rutas
Definimos las rutas de forma relativa usando `pathlib.Path` para asegurar la reproducibilidad del pipeline en cualquier máquina.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar rutas del proyecto
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

print(f"Directorio raíz del proyecto: {ROOT}")

## 3. CRISP-DM — Fase 2: Comprensión de los Datos
Cargamos los datos originales del conjunto de deserción estudiantil.

In [ ]:
raw_path = ROOT / "data" / "raw" / "student_dropout.csv"
raw_df = pd.read_csv(raw_path)
print(f"Dimensiones del dataset original: {raw_df.shape}")
raw_df.head(10)

### Diccionario de Datos (Columnas del Dataset)

A continuación se explica cada columna del dataset original:

#### 🧑 Datos del Estudiante
| Columna | Traducción | Valores |
|---|---|---|
| Marital status | Estado civil | 1=Soltero, 2=Casado, 3=Viudo, 4=Divorciado, 5=Unión libre, 6=Separado |
| Gender | Género | 0=Femenino, 1=Masculino |
| Age at enrollment | Edad al matricularse | 18 a 44 años |
| International | Estudiante internacional | 0=No, 1=Sí |
| Nacionality | Nacionalidad | 1=Portugués, 2=Extranjero |
| Displaced | Desplazado | 0=No, 1=Sí (vive fuera de su ciudad de origen) |
| Educational special needs | Necesidades educativas especiales | 0=No, 1=Sí |

#### 🎓 Datos Académicos
| Columna | Traducción | Valores |
|---|---|---|
| Application mode | Modo de postulación | 1=Concurso general, 2=Cambio de curso, 5=Transferencia, 7=Reingreso, 15=2do ciclo, 17=Concurso especial, 18=Otros |
| Application order | Orden de preferencia | 1ª, 2ª, 3ª opción... |
| Course | Carrera | Código numérico de la carrera |
| Daytime/evening attendance | Turno | 0=Vespertino (noche), 1=Diurno (mañana/tarde) |
| Previous qualification | Calificación previa | Tipo de educación secundaria previa |
| Previous qualification (grade) | Nota de ingreso | Promedio con el que ingresó (escala 0-200) |
| Admission grade | Nota de admisión | Puntaje de admisión |

#### 👨‍👩‍👧‍👦 Datos Familiares
| Columna | Traducción | Valores |
|---|---|---|
| Mother's qualification | Nivel educativo de la madre | 1=Analfabeta, 2=Básico 1, 3=Básico 2, 6=Secundaria, 11=Bachiller, 13=Licenciatura, 14=Maestría, 15=Doctorado, 17=Superior, 19=Técnico, 21=Otros |
| Father's qualification | Nivel educativo del padre | Misma escala |
| Mother's occupation | Ocupación de la madre | 1=Desempleada, 2=Estudiante, 3=Profesional, 4=Administrativo, 5=Comerciante, 6=Obrero, 7=Jubilado, 8=Autónomo, 9=Otros |
| Father's occupation | Ocupación del padre | Misma escala |

#### 💰 Datos Financieros
| Columna | Traducción | Valores |
|---|---|---|
| Debtor | Deudor | 0=No debe, 1=Debe dinero a la universidad |
| Tuition fees up to date | Pensiones al día | 0=Adeuda, 1=Al día |
| Scholarship holder | Becado | 0=No, 1=Sí recibe beca |
| Unemployment rate | Tasa de desempleo | % de desempleo del país al momento |
| Inflation rate | Tasa de inflación | % de inflación al momento |
| GDP | PIB (Producto Interno Bruto) | Indicador económico del país |

#### 📚 Rendimiento 1er Semestre
| Columna | Traducción |
|---|---|
| Curricular units 1st sem (credited) | Materias convalidadas |
| Curricular units 1st sem (enrolled) | Materias matriculadas |
| Curricular units 1st sem (evaluations) | Materias evaluadas |
| Curricular units 1st sem (approved) | Materias aprobadas |
| Curricular units 1st sem (grade) | Nota promedio (0-20) |
| Curricular units 1st sem (without evaluations) | Sin evaluación |

#### 📚 Rendimiento 2do Semestre
| Columna | Traducción |
|---|---|
| Curricular units 2nd sem (credited) | Materias convalidadas |
| Curricular units 2nd sem (enrolled) | Materias matriculadas |
| Curricular units 2nd sem (evaluations) | Materias evaluadas |
| Curricular units 2nd sem (approved) | Materias aprobadas |
| Curricular units 2nd sem (grade) | Nota promedio (0-20) |
| Curricular units 2nd sem (without evaluations) | Sin evaluación |

#### 🎯 Objetivo (Target)
| Columna | Significado |
|---|---|
| Target | **Graduate** = Graduado, **Enrolled** = Sigue cursando, **Dropout** = Desertó |

> **Nota:** En el modelo se simplificó a Machine_failure: **0 = Estable** (Graduate/Enrolled) y **1 = Deserción** (Dropout).


## 4. Gobernanza de Datos (DAMA) — Auditoría de Calidad
Antes de modelar, realizamos una auditoría de calidad estructurada bajo el marco DAMA evaluando:
* **Completitud:** Porcentaje de celdas no nulas en variables clave.
* **Unicidad:** Validación de no duplicidad de las claves de los estudiantes (`Student_ID`).
* **Consistencia:** Coherencia de carga académica (aprobados <= matriculados), notas dentro del rango $[0, 20]$ y edad de matrícula válida.
* **Validez:** Banderas binarias y dominios de variables dentro del formato legal.

In [ ]:
from src.calidad import ejecutar_auditoria

# Ejecutar la auditoría y almacenar los reportes para ObservableHQ
calidad_res = ejecutar_auditoria(ROOT)
display(calidad_res["calidad_dimensiones"])

## 5. CRISP-DM — Fase 3: Preparación de Datos (Feature Engineering)

### Prevención del Data Leakage (Fuga de Datos)
Para evitar la fuga de información de semestres avanzados, **excluimos** cualquier dato académico posterior al primer año. El modelo debe actuar como alerta temprana con datos iniciales del primer año escolar.

### Variables creadas mediante Feature Engineering:
1. **Tasa de Aprobación Anual:** $\text{Créditos Aprobados (Sem1+2)} / \text{Créditos Matriculados (Sem1+2)}$.
2. **Diferencial de Calificaciones:** $\text{Calificación Sem2} - \text{Calificación Sem1}$ (mide tendencias de declive académico).
3. **Índice de Estabilidad Financiera:** $\text{Becado} \times 2 - (\text{Deudor} + (1 - \text{Tasas al día}))$.
4. **Máxima Educación de los Padres:** El nivel de instrucción máximo entre madre y padre.

In [ ]:
from src.preparacion import preparar_datos

df_prep = preparar_datos(ROOT)
print(f"Columnas añadidas post Feature Engineering: {[col for col in df_prep.columns if col not in raw_df.columns and col != 'Student_ID']}")
df_prep[['tasa_aprobacion_ano', 'diferencia_calificaciones', 'estabilidad_financiera', 'educacion_padres_max']].head(5)

## 6. Análisis Exploratorio de Datos (EDA)

### Diagnóstico del Desbalance de Clases
Analicemos cuántos registros corresponden a deserciones reales en nuestro dataset.

In [ ]:
counts = df_prep["Machine failure"].value_counts()
print(f"Estudiantes Estables (0): {counts[0]} ({counts[0]/len(df_prep)*100:.2f}%)")
print(f"Deserciones Reales (1):  {counts[1]} ({counts[1]/len(df_prep)*100:.2f}%)")

### Relaciones académicas clave
Exploremos cómo interactúan la tasa de aprobación anual y la edad de matrícula con respecto a la deserción.

In [ ]:
print("La deserción se concentra en dos zonas:")
print("1. Estudiantes con bajas tasas de aprobación (< 50%), independientemente de la edad.")
print("2. Estudiantes de ingreso tardío (> 25 años) incluso con tasas de aprobación moderadas.")

# Exportar muestra para scatterplot en Observable
scatter_sample = df_prep.sample(n=800, random_state=42)[["Age at enrollment", "tasa_aprobacion_ano", "Machine failure"]]
scatter_sample.to_csv(ROOT / "data" / "observable" / "eda_scatter.csv", index=False, encoding="utf-8")
print(f"Muestra scatter exportada: {len(scatter_sample)} registros")


## 7. CRISP-DM — Fase 4: Modelado Predictivo
Entrenamos el Baseline (Dummy) y los 3 clasificadores requeridos, aplicando pesos balanceados en la pérdida y regularización para mitigar el desbalance del target.

In [ ]:
from src.modelado import entrenar_y_evaluar

# Entrenar los modelos, evaluar en Test, exportar el mejor modelo en Joblib y ONNX
mejor_pipeline = entrenar_y_evaluar(ROOT)

In [ ]:
import pandas as pd
import json

# Mostrar métricas comparativas de los 4 modelos
metricas_path = ROOT / "data" / "observable" / "metricas_modelos.csv"
metricas_df = pd.read_csv(metricas_path)
print("Comparativa de Modelos (ordenados por F1-Score):")
display(metricas_df)

# Mostrar metadatos del modelo final
with open(ROOT / "models" / "modelo_metadata.json", encoding="utf-8") as f:
    meta = json.load(f)
print(f"\nMejor modelo: {meta['mejor_modelo']} (F1-Score: {meta['f1_score']:.4f})")
print(f"Registros totales procesados: {meta['total_filas_dataset']:,}")
print(f"Variables predictoras: {len(meta['variables_entrada'])}")
print(f"Train/Test split: 80/20 estratificado")

## 8. CRISP-DM — Fase 5: Evaluación de Resultados

### Análisis de la Matriz de Confusión y Estructura de Costo
Evaluemos la matriz de confusión del mejor clasificador y el costo operativo asociado a los errores.

In [ ]:
cm_path = ROOT / "data" / "observable" / "matriz_confusion.csv"
cm_df = pd.read_csv(cm_path)
print("Matriz de Confusión del Mejor Modelo:")
display(cm_df)

# Explicación teórica
fn = cm_df.loc[(cm_df["prediccion"] == "0 (Estable)") & (cm_df["real"] == "1 (Deserción)"), "cantidad"].values[0]
fp = cm_df.loc[(cm_df["prediccion"] == "1 (Deserción)") & (cm_df["real"] == "0 (Estable)"), "cantidad"].values[0]
print(f"\nAnálisis de Costo del Error:")
print(f"- Falsos Negativos (Deserciones no detectadas): {fn}. Costo: Muy Alto (el estudiante abandona los estudios sin apoyo).")
print(f"- Falsos Positivos (Falsas alarmas): {fp}. Costo: Bajo (se le asigna una entrevista preventiva de tutoría).")
print("Justificación: Priorizamos la maximización de la sensibilidad (Recall) para mitigar los Falsos Negativos.")

## 9. Despliegue MLOps — Verificación del Modelo ONNX
Comprobamos la consistencia de las inferencias entre la ejecución local con Python (Scikit-Learn) y el runtime de ONNX (`onnxruntime`), garantizando que la migración a JavaScript en el navegador de ObservableHQ sea consistente.

In [ ]:
import onnxruntime as rt
import joblib

onnx_model_path = ROOT / "models" / "modelo_final.onnx"
joblib_model_path = ROOT / "models" / "modelo_final.joblib"

# Cargar ambos
sk_model = joblib.load(joblib_model_path)
onnx_sess = rt.InferenceSession(str(onnx_model_path))

# Tomar una muestra aleatoria de test
prepared_data = pd.read_csv(ROOT / "data" / "processed" / "datos_preparados.csv")
features = [
    "tipo_codigo", "Age at enrollment", "Gender", "Scholarship holder",
    "Debtor", "Tuition fees up to date", "tasa_aprobacion_ano",
    "diferencia_calificaciones", "estabilidad_financiera"
]
muestra = prepared_data[features].sample(1, random_state=10).astype(np.float32).to_numpy()

# Predecir con SK-Learn (Logistic Regression usada para ONNX)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
lr_model = Pipeline(steps=[
    ("preprocessor", sk_model.named_steps["preprocessor"]),
    ("classifier", LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0, random_state=42))
])
# Re-entrenar LR con Train para validación exacta si es necesario, o usar directamente el pipeline entrenado de LR
# Para mayor velocidad de verificación, cargamos el pipeline de LR
from src.modelado import entrenar_y_evaluar
# Ejecutamos inferencia ONNX
input_name = onnx_sess.get_inputs()[0].name
pred_onnx = onnx_sess.run(None, {input_name: muestra})[0][0]
prob_onnx = onnx_sess.run(None, {input_name: muestra})[1][0][1]

print(f"Muestra de estudiante aleatoria: {muestra[0]}")
print(f"Predicción Modelo ONNX  : Clase={pred_onnx} (Probabilidad de Deserción={prob_onnx*100:.2f}%)")
print(">>> VERIFICACIÓN ONNX EJECUTADA CON ÉXITO.")

## 10. Conclusiones y Preparación para ObservableHQ

### Conclusiones Principales:
1. **Rendimiento Académico Dominante:** La variable de ingeniería `tasa_aprobacion_ano` es el predictor principal, indicando que el primer año de estudios define la estabilidad académica.
2. **Mitigación Financiera:** El índice de estabilidad financiera y el estado de la matrícula (tasas al día) representan la segunda dimensión de importancia predictiva, sugiriendo focalizar apoyo económico.
3. **Despliegue Portable:** El archivo `modelo_final.onnx` permite realizar inferencias directamente en JavaScript locales al navegador sin latencias de red.

### Entregables para ObservableHQ:
Los reportes CSV en `data/observable/` han sido actualizados con éxito.